# RockID · Phase 1 · Dataset audit

Checks the Kaggle rocks-and-minerals dataset **before any training**: class counts, duplicates and near-duplicates, image quality, background bias and license. It then builds train/val/test splits with no leakage between them and writes a report.

**Before you run** (see `docs/phase-0-setup.md`):
1. **Runtime → Change runtime type → T4 GPU** (embeddings are slow on CPU).
2. Colab **Secrets** (key icon, left sidebar): `KAGGLE_USERNAME`, `KAGGLE_KEY`, plus `GITHUB_TOKEN` if the repo is private. Turn on notebook access for each.
3. Code location: set `REPO_URL` below, **or** upload the repo's `ml/` folder to `MyDrive/RockID/code/ml`.
4. *(Recommended)* Field photos in `MyDrive/RockID/field_test/<class name>/`.

Every slow step caches to Drive, so after a disconnect, **Runtime → Run all** picks up where it stopped.

**Outputs** go to `MyDrive/RockID/audit/<timestamp>/`. The `review/` contact sheets contain dataset images. Look at them, but **don't commit them**.

## 1 · Settings

In [ ]:
REPO_URL = "https://github.com/GrowingRui/RockID.git"   # empty = use MyDrive/RockID/code/ml instead
REPO_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/RockID"
RUN_EMBEDDINGS = True    # near-duplicate search by visual similarity + probes. Needs the GPU runtime.

## 2 · Setup: Drive, code, dependencies

In [ ]:
import importlib, json, os, pathlib, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = pathlib.Path(DRIVE_ROOT)
CACHE = DRIVE / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

if REPO_URL:
    CODE = pathlib.Path("/content/rockid")
    url = REPO_URL
    try:
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass  # public repo, no token needed
    cmd = ["git", "-C", str(CODE), "pull", "--ff-only"] if CODE.exists() else \
          ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, str(CODE)]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        raise RuntimeError("git clone/pull failed. Check REPO_URL, REPO_BRANCH and the GITHUB_TOKEN secret.") from None
    ML = CODE / "ml"
else:
    ML = DRIVE / "code" / "ml"

assert (ML / "rockid_ml").exists(), f"rockid_ml not found under {ML}"
sys.path.insert(0, str(ML))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash>=4.3"], check=True)

import numpy as np, pandas as pd
import rockid_ml
from rockid_ml import audit, dedup, embed, images, probes, report, scan, split
from rockid_ml.config import AuditConfig
for m in (images, scan, dedup, embed, split, probes, audit, report):
    importlib.reload(m)  # pick up code edits without restarting the runtime

cfg = AuditConfig()
print("rockid_ml", rockid_ml.__version__, "from", ML)

## 3 · Download the dataset (cached on Drive)

In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

WORK = pathlib.Path("/content/data"); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "raw"
ZIP_CACHE = CACHE / "dataset.zip"

if not ZIP_CACHE.exists():
    subprocess.run(["kaggle", "datasets", "download", "-d", cfg.kaggle_dataset, "-p", str(WORK), "--force"], check=True)
    shutil.copy(next(WORK.glob("*.zip")), ZIP_CACHE)
if not DATA.exists():
    with zipfile.ZipFile(ZIP_CACHE) as z:
        z.extractall(DATA)  # zipfile drops absolute paths and ".." entries

license_info = None
try:
    subprocess.run(["kaggle", "datasets", "metadata", "-p", str(WORK), cfg.kaggle_dataset], check=True, capture_output=True)
    license_info = json.loads((WORK / "dataset-metadata.json").read_text())
    print("License:", [l.get("name") for l in license_info.get("licenses", [])])
except Exception as e:
    print("Could not read Kaggle metadata, check the license by hand:", e)

## 4 · Scan folders and labels

In [ ]:
df = scan.scan_dataset(DATA)
label_files = scan.find_label_files(DATA)
raw_names = scan.raw_name_map(df)

print(f"{len(df):,} images · {df['label'].nunique()} classes (from {df['label_raw'].nunique()} folder names)")
print("Pre-made split folders found:", df["source_split"].value_counts(dropna=False).to_dict())
print("Label-like files:", label_files[:10] or "none")
if (df["label"] == "unlabeled").mean() > 0.05:
    print("WARNING: many images are not inside a class folder. Inspect the structure below before going on.")
print("\nTop-level folders:", df["rel_path"].str.split("/").str[0].value_counts().to_dict())
df["label"].value_counts().to_frame("images")

## 5 · Per-image stats: decode, size, blur, hashes, background (resumable)

In [ ]:
df = audit.analyze_all(DATA, df, cache_csv=CACHE / "image_stats.csv", plain_tol=cfg.plain_tol)
valid_mask = df["decode_error"].isna()
valid = df[valid_mask].reset_index(drop=True)
print(f"decoded {len(valid):,} / {len(df):,} · decode errors: {(~valid_mask).sum()}")
df.loc[~valid_mask, ["rel_path", "decode_error"]].head(10)

## 6 · Visual embeddings (GPU, cached)

In [ ]:
emb = emb_border = None
if RUN_EMBEDDINGS:
    import tensorflow as tf
    print("TF", tf.__version__, "· GPU:", tf.config.list_physical_devices("GPU") or "NONE, this will be slow")
    paths = [str(DATA / p) for p in valid["rel_path"]]
    emb = audit.cached_embeddings(CACHE / "emb_full.npz", valid["rel_path"],
                                  lambda: embed.embed_paths(paths, size=cfg.embed_size))
    if cfg.background_probe:
        emb_border = audit.cached_embeddings(CACHE / "emb_border.npz", valid["rel_path"],
                                             lambda: embed.embed_paths(paths, size=cfg.embed_size, variant="border"))
    print("embeddings:", emb.shape)

## 7 · Duplicates and near-duplicates

In [ ]:
pairs = audit.near_duplicate_pairs(valid, emb, cfg)
df["cluster_id"] = audit.assign_clusters(df, valid_mask, pairs)
sizes = df[valid_mask].groupby("cluster_id").size()
exact_groups = int((valid.groupby("pixel_sha256").size() > 1).sum())
dup_stats = {
    "Exact-duplicate groups": exact_groups,
    "Near-duplicate pairs (pHash)": int((pairs["kind"] == "phash").sum()),
    "Near-duplicate pairs (embedding)": int((pairs["kind"] == "embedding").sum()),
    "Clusters with more than one image": int((sizes > 1).sum()),
    "Images in such clusters": int(sizes[sizes > 1].sum()),
    "Largest cluster": int(sizes.max()),
}
dup_stats

## 8 · Class decisions, selection and split

Leave these empty on the first run. Once you've reviewed the report and contact sheets (step 12), fill them in and **re-run from this cell down** (*Runtime → Run cell and below*). Use class **slugs** as shown in the class table.

In [ ]:
from dataclasses import replace
CLASS_MERGES = {}   # e.g. {"smoky-quartz": "quartz"}
DROP_CLASSES = []   # e.g. ["unlabeled"]
cfg = replace(cfg, class_merges=CLASS_MERGES, drop_classes=DROP_CLASSES)

d = audit.select_and_split(df, cfg)
kept = d[d["split"].notna()]
leaks = split.leaking_clusters(kept)
assert leaks.empty, f"{len(leaks)} clusters span several splits"
print(f"kept {len(kept):,} images in {kept['label_final'].nunique()} classes · leakage check passed")
print(d["excluded_reason"].value_counts().to_dict())
kept["split"].value_counts().to_frame("images")

## 9 · Linear probe and background probe

In [ ]:
probe = bg_probe = nearest = confused = None
if emb is not None:
    pos = pd.Series(np.arange(len(valid)), index=valid["rel_path"])
    rows = pos[kept["rel_path"]].to_numpy()
    y, groups = kept["label_final"].to_numpy(), kept["cluster_id"].to_numpy()
    probe = probes.linear_probe(emb[rows], y, groups, seed=cfg.seed)
    nearest = probes.nearest_classes(emb[rows], y)
    confused = probes.confused_pairs(y, probe["pred"])
    print(f"probe top-1 {probe['top1']:.1%} · top-3 {probe['top3']:.1%} · chance {probe['chance']:.1%}")
    if emb_border is not None:
        bg_probe = probes.linear_probe(emb_border[rows], y, groups, seed=cfg.seed)
        print(f"background-only probe top-1 {bg_probe['top1']:.1%}")
confused

## 10 · Class table and recommendations

In [ ]:
t = audit.class_table(d, cfg, probe, nearest)
n_cross = int(d.loc[d["cross_class"], "cluster_id"].nunique())
recs = audit.recommendations(t, cfg, probe, bg_probe, n_cross)
for r in recs:
    print("•", r)
t

## 11 · Field test set

In [ ]:
FIELD = DRIVE / "field_test"
field = None
if FIELD.exists() and any(FIELD.iterdir()):
    embed_fn = (lambda ps: embed.embed_paths(ps, size=cfg.embed_size)) if emb is not None else None
    field = audit.check_field_set(FIELD, valid, emb, cfg, embed_fn)
    print(f"{len(field)} field images · {int(field['leaks_into_dataset'].sum())} also in the dataset · "
          f"{int((~field['known_class']).sum())} unknown class")
else:
    print("No field set yet at", FIELD)

## 12 · Report, figures and contact sheets

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")
OUT = DRIVE / "audit" / stamp
FIG, REV = OUT / "figures", OUT / "review"
FIG.mkdir(parents=True); REV.mkdir()
rng = np.random.default_rng(cfg.seed)
clean = d[d["excluded_reason"].isna()]

figures = {k: FIG / f"{k}.png" for k in ("class_counts", "plain_bg", "blur", "short_side")}
report.plot_class_counts(t, figures["class_counts"], cfg.min_images_per_class)
report.plot_share(t, "plain_bg_share", figures["plain_bg"], "share of plain-background images", cfg.studio_class_warn)
report.plot_hist(valid["blur_var"], figures["blur"], "Laplacian variance (higher = sharper)", cfg.blur_low, log=True)
report.plot_hist(np.minimum(valid["width"], valid["height"]), figures["short_side"], "short side (px)", cfg.min_side)

sheets = []
for cls, g in clean.groupby("label_final"):
    sample = g.sample(min(24, len(g)), random_state=cfg.seed)
    sheets.append(report.contact_sheet([(str(DATA / p), cls) for p in sample["rel_path"]], REV / "classes" / f"{cls}.jpg"))

multi = sizes[sizes > 1].sort_values(ascending=False)
sheets.append(report.contact_sheet(report.cluster_rows(d, DATA, multi.index[:40]), REV / "clusters_largest.jpg", rows_of=6))
cross_ids = d.loc[d["cross_class"], "cluster_id"].unique()[:40]
sheets.append(report.contact_sheet(report.cluster_rows(d, DATA, cross_ids), REV / "cross_class.jpg", rows_of=6))

# Pairs just inside the thresholds: are these really the same photo?
edge = pairs[((pairs["kind"] == "phash") & (pairs["score"] >= cfg.phash_max_dist - 1)) |
             ((pairs["kind"] == "embedding") & (pairs["score"] < cfg.emb_min_sim + 0.01))]
edge = edge.sample(min(30, len(edge)), random_state=cfg.seed)
items = []
for r in edge.itertuples():
    a, b = valid.iloc[int(r.i)], valid.iloc[int(r.j)]
    items += [(str(DATA / a.rel_path), f"{r.kind} {r.score:.2f}"), (str(DATA / b.rel_path), b.label)]
sheets.append(report.contact_sheet(items, REV / "threshold_edge_pairs.jpg", cols=6))

blurry = valid.nsmallest(48, "blur_var")
sheets.append(report.contact_sheet([(str(DATA / r.rel_path), f"{r.blur_var:.0f}") for r in blurry.itertuples()], REV / "blurriest.jpg"))
ranked = valid.sort_values("border_plain_frac")
bg_items = [(str(DATA / r.rel_path), f"plain {r.border_plain_frac:.2f}") for r in pd.concat([ranked.tail(24), ranked.head(24)]).itertuples()]
sheets.append(report.contact_sheet(bg_items, REV / "background_most_vs_least_plain.jpg"))
sheets = [s for s in sheets if s]

below = t.index[(t["status"] == "below_min")].tolist()
summary = {
    "Images found": f"{len(df):,}",
    "Classes": f"{df['label'].nunique()} (from {df['label_raw'].nunique()} folder names) → {kept['label_final'].nunique()} kept",
    "Decode errors": int((~valid_mask).sum()),
    "Clean images": f"{len(kept):,} ({len(kept)/len(df):.0%})",
    "Excluded": d["excluded_reason"].value_counts().to_dict(),
    "Classes below minimum": ", ".join(below) or "none",
    "Splits": kept["split"].value_counts().to_dict(),
    "Leakage check": "passed (no duplicate cluster spans two splits)",
    "Field set": "missing" if field is None else f"{int((~field['leaks_into_dataset'] & field['known_class']).sum())} usable images",
}
path = report.write_report(OUT, cfg.to_dict(), summary, t, license_info, label_files, raw_names, dup_stats,
                           probe, bg_probe, confused, field, recs, sheets, figures)

audit.manifest(d).to_csv(OUT / "split_manifest.csv", index=False)
t.to_csv(OUT / "class_table.csv")
pairs.assign(path_i=valid["rel_path"].to_numpy()[pairs["i"]], path_j=valid["rel_path"].to_numpy()[pairs["j"]]).to_csv(OUT / "near_dup_pairs.csv", index=False)
if field is not None:
    field[["rel_path", "label_final", "leaks_into_dataset", "known_class"]].to_csv(OUT / "field_manifest.csv", index=False)
print("Report:", path)

## 13 · Download the files that belong in the repo

In [ ]:
from google.colab import files
bundle = WORK / f"rockid_audit_{stamp}.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(OUT / "audit.md", f"reports/audit-{stamp}/audit.md")
    for f in FIG.glob("*.png"):
        z.write(f, f"reports/audit-{stamp}/figures/{f.name}")
    z.write(OUT / "class_table.csv", f"reports/audit-{stamp}/class_table.csv")
    z.write(OUT / "split_manifest.csv", "splits/split_manifest.csv")
    if (OUT / "field_manifest.csv").exists():
        z.write(OUT / "field_manifest.csv", "splits/field_manifest.csv")
files.download(str(bundle))
print("Unzip into the repo's ml/ folder. review/ stays on Drive only.")

## Next
1. Read `audit.md` and the `review/` sheets.
2. Decide merges and drops, update step 8, and re-run from step 8.
3. Share the report summary for approval of the final class list. Phase 2 (training) starts after that.